# 📊 Trích Xuất & Tổng Hợp Dữ Liệu Milestone 2 Toàn Nhóm (Hoàng — Infrastructure & Statistics)

> **Mục đích:** Xử lý trực tiếp toàn bộ kho dữ liệu Milestone 2 trên Google Drive (`arXiv-data/milestone 2/`) của **cả 5 thành viên (Hoàng, Huỳnh Nhi, Khang, Mẫn Nhi, Trọng)**, tổng hợp các chỉ số toàn nhóm và phân rã theo từng owner để đưa vào **Mục 2.3 (Thống kê)** và **Mục 2.4 (Benchmark)** của báo cáo `arvix-report` mà **không cần tải hàng chục GB dữ liệu thô về máy cục bộ**.

**Đầu ra của Notebook:**
1. `m2_hierarchy_summary.json`: Tóm tắt các chỉ số thống kê M2 cho **toàn nhóm** (~25.000 bài) kèm bảng phân rã chi tiết theo **5 Owner**.
2. `m2_hierarchy_publications.csv`: Bảng tóm tắt theo từng bài báo của cả nhóm.
3. `m2-hoang-elements.png` & `m2-hoang-versions.png`: Hai biểu đồ phân tích trực quan toàn nhóm độ phân giải cao.
4. `m2_hoang_group_summary.zip`: Gói nén tự động tải về chỉ ~1–2 MB.

## Bước 1: Kết nối Google Drive và Cài đặt Thư viện

In [ ]:
# Mount Google Drive
from google.colab import drive
import os, sys, json, csv, glob, sqlite3, shutil, zipfile
from pathlib import Path
from google.colab import files

drive.mount('/content/drive')

# Cài đặt thư viện xử lý dữ liệu
!pip install -q pyarrow pandas matplotlib

## Bước 2: Dò tìm Thư mục Gốc `milestone 2` của Cả Nhóm trên Drive

In [ ]:
possible_roots = [
    '/content/drive/MyDrive/arXiv-data/milestone 2',
    '/content/drive/Shareddrives/arXiv-data/milestone 2',
    '/content/drive/MyDrive/milestone 2',
]

# Tìm trong các phím tắt shortcut
possible_roots.extend(glob.glob('/content/drive/.shortcut-targets-by-id/*/arXiv-data/milestone 2'))
possible_roots.extend(glob.glob('/content/drive/.shortcut-targets-by-id/*/milestone 2'))

m2_root = None
for p in possible_roots:
    if os.path.exists(p) and os.path.isdir(p):
        m2_root = p
        break

if m2_root is None:
    print('⚠️ Chưa tự động nhận diện được đường dẫn mặc định.')
    m2_root = input('Vui lòng nhập đường dẫn thư mục milestone 2 trên Drive của bạn: ').strip()

print(f'✅ Đã xác định thư mục gốc Milestone 2: {m2_root}')

# Kiểm tra các thư mục thành viên
EXPECTED_OWNERS = ['hoang', 'huynh-nhi', 'khang', 'man-nhi', 'trong']
owner_dirs = {}

for item in os.listdir(m2_root):
    item_path = os.path.join(m2_root, item)
    if os.path.isdir(item_path):
        norm = item.lower().replace('_', '-')
        for exp in EXPECTED_OWNERS:
            if exp in norm or norm in exp:
                owner_dirs[exp] = item_path

print(f'Phát hiện {len(owner_dirs)}/5 thư mục thành viên:')
for o, p in owner_dirs.items():
    print(f'  - [{o}]: {p}')

## Bước 3: Hàm Quét Dữ Liệu Từng Thành Viên (Hỗ trợ Parquet & SQLite chuẩn)

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

def find_best_sqlite_file(folder: str) -> str | None:
    """Tìm file SQLite tốt nhất, ưu tiên kết quả chính thức 'parsing.sqlite' và loại bỏ snapshot dở dang."""
    candidates = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith('.sqlite') or f.endswith('.db'):
                full_path = os.path.join(root, f)
                size = os.path.getsize(full_path)
                score = 0
                if 'parsing.sqlite' in f.lower():
                    score += 1000
                if 'artifacts' in root.lower() or 'result' in root.lower():
                    score += 500
                if 'snapshot' in root.lower() or 'checkpoint' in f.lower():
                    score -= 300
                candidates.append((score, size, full_path))
    if not candidates:
        return None
    # Sắp xếp theo score cao nhất, dung lượng lớn nhất
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    best_file = candidates[0][2]
    print(f'  -> Chọn file SQLite tối ưu: {best_file} ({candidates[0][1]/(1024*1024):.2f} MB)')
    return best_file

def process_owner_data(owner_name: str, folder_path: str):
    """Đọc dữ liệu của một owner dù lưu ở định dạng Parquet hay SQLite."""
    print(f'\n⏳ Đang xử lý dữ liệu của owner: [{owner_name}] tại {folder_path}...')
    
    # 1. Kiểm tra xem có file Parquet không
    parquet_files = ['papers.parquet', 'hierarchy.parquet', 'elements.parquet', 'bibtex.parquet']
    has_parquet = all(os.path.exists(os.path.join(folder_path, f)) for f in parquet_files)
    
    if has_parquet:
        print(f'  -> Phát hiện định dạng PARQUET.')
        df_p = pq.read_table(os.path.join(folder_path, 'papers.parquet')).to_pandas()
        df_p['version_count'] = 1 + df_p['revised_dates'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
        
        t_hier = pq.read_table(os.path.join(folder_path, 'hierarchy.parquet'), columns=['arxiv_id', 'version', 'element_id'])
        df_h = t_hier.to_pandas()
        
        t_elem = pq.read_table(os.path.join(folder_path, 'elements.parquet'), columns=['arxiv_id', 'element_id', 'kind'])
        df_e = t_elem.to_pandas()
        
        t_bib = pq.read_table(os.path.join(folder_path, 'bibtex.parquet'), columns=['arxiv_id', 'citation_key', 'entry_type'])
        df_bib = t_bib.to_pandas()
        
        parsed_vers = df_h[['arxiv_id', 'version']].drop_duplicates().groupby('arxiv_id').size().rename('parsed_version_count')
        edge_counts = df_h.groupby('arxiv_id').size().rename('hierarchy_edge_count')
        dedup_elem_counts = df_e.groupby('arxiv_id').size().rename('dedup_element_count')
        
        df_m = df_p[['arxiv_id', 'version_count']].set_index('arxiv_id')
        df_m = df_m.join(parsed_vers).fillna({'parsed_version_count': 0})
        df_m['parsed_version_count'] = df_m['parsed_version_count'].astype(int)
        df_m['empty_version_count'] = df_m['version_count'] - df_m['parsed_version_count']
        df_m = df_m.join(edge_counts).fillna({'hierarchy_edge_count': 0})
        df_m['hierarchy_edge_count'] = df_m['hierarchy_edge_count'].astype(int)
        df_m['raw_element_count'] = df_m['hierarchy_edge_count']
        df_m = df_m.join(dedup_elem_counts).fillna({'dedup_element_count': 0})
        df_m['dedup_element_count'] = df_m['dedup_element_count'].astype(int)
        df_m['owner'] = owner_name
        
        pub_rows = df_m.reset_index()[['arxiv_id', 'owner', 'version_count', 'empty_version_count', 'parsed_version_count', 'raw_element_count', 'dedup_element_count', 'hierarchy_edge_count']]
        kind_counts = df_e['kind'].value_counts().to_dict()
        
        storage_bytes = sum(os.path.getsize(os.path.join(folder_path, f)) for f in os.listdir(folder_path) if f.endswith('.parquet'))
        
        return {
            'owner': owner_name,
            'pub_rows': pub_rows,
            'total_publications': len(df_p),
            'declared_versions': int(df_p['version_count'].sum()),
            'parsed_versions': int(df_m['parsed_version_count'].sum()),
            'empty_versions': int(df_m['empty_version_count'].sum()),
            'raw_elements': len(df_h),
            'dedup_elements': len(df_e),
            'hierarchy_edges': len(df_h),
            'bibtex_entries': len(df_bib),
            'kind_counts': kind_counts,
            'storage_mb': round(storage_bytes / (1024 * 1024), 2)
        }
        
    # 2. Kiểm tra định dạng SQLite
    sqlite_path = find_best_sqlite_file(folder_path)
    if sqlite_path:
        con = sqlite3.connect(sqlite_path)
        try:
            cur = con.cursor()
            
            # Kiểm tra cấu trúc các cột trong publications_meta linh hoạt
            cols = [c[1] for c in cur.execute('PRAGMA table_info(publications_meta)').fetchall()]
            
            # Query an toàn cho publications_meta
            if 'empty_versions' in cols:
                pubs = cur.execute('SELECT arxiv_id, version_count, empty_versions, raw_element_count FROM publications_meta ORDER BY arxiv_id').fetchall()
                has_empty_json = True
            elif 'empty_version_count' in cols:
                pubs = cur.execute('SELECT arxiv_id, version_count, empty_version_count, raw_element_count FROM publications_meta ORDER BY arxiv_id').fetchall()
                has_empty_json = False
            else:
                pubs = cur.execute('SELECT arxiv_id, version_count, 0, raw_element_count FROM publications_meta ORDER BY arxiv_id').fetchall()
                has_empty_json = False
                
            elem_counts = dict(cur.execute('SELECT arxiv_id, COUNT(*) FROM elements GROUP BY arxiv_id').fetchall())
            edge_counts = dict(cur.execute('SELECT arxiv_id, COUNT(*) FROM hierarchy GROUP BY arxiv_id').fetchall())
            
            pub_rows_list = []
            for row in pubs:
                aid = row[0]
                vcnt = int(row[1] or 0)
                raw_cnt = int(row[3] or 0)
                
                if has_empty_json:
                    try:
                        empty_list = json.loads(row[2] or '[]')
                        ecnt = len(empty_list)
                    except Exception:
                        ecnt = 0
                else:
                    ecnt = int(row[2] or 0)
                    
                pcnt = max(0, vcnt - ecnt)
                pub_rows_list.append({
                    'arxiv_id': str(aid),
                    'owner': owner_name,
                    'version_count': vcnt,
                    'empty_version_count': ecnt,
                    'parsed_version_count': pcnt,
                    'raw_element_count': raw_cnt,
                    'dedup_element_count': elem_counts.get(aid, 0),
                    'hierarchy_edge_count': edge_counts.get(aid, 0)
                })
            pub_rows = pd.DataFrame(pub_rows_list)
            
            kind_counts = dict(cur.execute('SELECT kind, COUNT(*) FROM elements GROUP BY kind').fetchall())
            dedup_cnt = cur.execute('SELECT COUNT(*) FROM elements').fetchone()[0]
            edge_cnt = cur.execute('SELECT COUNT(*) FROM hierarchy').fetchone()[0]
            
            # Kiểm tra bảng bibtex
            has_bib = cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='bibtex'").fetchone()
            bib_cnt = cur.execute('SELECT COUNT(*) FROM bibtex').fetchone()[0] if has_bib else 0
            
            db_size_mb = round(os.path.getsize(sqlite_path) / (1024 * 1024), 2)
            
            return {
                'owner': owner_name,
                'pub_rows': pub_rows,
                'total_publications': len(pub_rows),
                'declared_versions': int(pub_rows['version_count'].sum()),
                'parsed_versions': int(pub_rows['parsed_version_count'].sum()),
                'empty_versions': int(pub_rows['empty_version_count'].sum()),
                'raw_elements': int(pub_rows['raw_element_count'].sum()),
                'dedup_elements': dedup_cnt,
                'hierarchy_edges': edge_cnt,
                'bibtex_entries': bib_cnt,
                'kind_counts': kind_counts,
                'storage_mb': db_size_mb
            }
        finally:
            con.close()
            
    print(f'  ⚠️ Không tìm thấy Parquet hoặc SQLite hợp lệ trong thư mục của {owner_name}.')
    return None

## Bước 4: Chạy Tổng Hợp Toàn Nhóm (5 Owner)

In [ ]:
all_owner_results = []
all_pub_dfs = []
combined_kind_counts = {}

for owner_name, folder_path in owner_dirs.items():
    res = process_owner_data(owner_name, folder_path)
    if res:
        all_owner_results.append(res)
        all_pub_dfs.append(res['pub_rows'])
        for k, v in res['kind_counts'].items():
            combined_kind_counts[k] = combined_kind_counts.get(k, 0) + v

# Ghép DataFrame toàn nhóm
if all_pub_dfs:
    df_all_pubs = pd.concat(all_pub_dfs, ignore_index=True)
else:
    raise RuntimeError('Không đọc được dữ liệu của owner nào!')

# Tổng hợp số liệu toàn nhóm
total_pubs = len(df_all_pubs)
total_declared = int(df_all_pubs['version_count'].sum())
total_parsed = int(df_all_pubs['parsed_version_count'].sum())
total_empty = int(df_all_pubs['empty_version_count'].sum())
total_raw_elem = int(df_all_pubs['raw_element_count'].sum())
total_dedup_elem = sum(r['dedup_elements'] for r in all_owner_results)
total_edges = sum(r['hierarchy_edges'] for r in all_owner_results)
total_bib = sum(r['bibtex_entries'] for r in all_owner_results)
total_storage = sum(r['storage_mb'] for r in all_owner_results)

dedup_saved = total_raw_elem - total_dedup_elem
dedup_ratio = round(dedup_saved / total_raw_elem * 100, 2) if total_raw_elem > 0 else 0.0

# Bảng phân rã theo 5 Owner
owner_breakdown = []
for r in all_owner_results:
    o_saved = r['raw_elements'] - r['dedup_elements']
    o_ratio = round(o_saved / r['raw_elements'] * 100, 2) if r['raw_elements'] > 0 else 0.0
    owner_breakdown.append({
        'owner': r['owner'],
        'publications': r['total_publications'],
        'declared_versions': r['declared_versions'],
        'parsed_versions': r['parsed_versions'],
        'empty_versions': r['empty_versions'],
        'raw_elements': r['raw_elements'],
        'dedup_elements': r['dedup_elements'],
        'dedup_ratio_pct': o_ratio,
        'hierarchy_edges': r['hierarchy_edges'],
        'bibtex_entries': r['bibtex_entries'],
        'storage_mb': r['storage_mb']
    })

group_summary = {
    'is_mock': False,
    'total_publications': total_pubs,
    'declared_versions': total_declared,
    'parsed_versions': total_parsed,
    'empty_versions': total_empty,
    'parsing_success_rate_pct': round(total_parsed / total_declared * 100, 2) if total_declared > 0 else 0.0,
    'raw_element_count_sum': total_raw_elem,
    'dedup_element_count': total_dedup_elem,
    'dedup_saved_count': dedup_saved,
    'dedup_ratio_pct': dedup_ratio,
    'hierarchy_edge_count': total_edges,
    'total_bibtex_entries': total_bib,
    'total_storage_mb': round(total_storage, 2),
    'element_kind_counts': combined_kind_counts,
    'owner_breakdown': owner_breakdown
}

print('\n' + '='*55)
print('🎉 KẾT QUẢ TỔNG HỢP TOÀN NHÓM (MILESTONE 2):')
print(f"  - Tổng số bài báo: {total_pubs:,} bài")
print(f"  - Tổng phiên bản: {total_declared:,} (Có cây: {total_parsed:,}, Rỗng: {total_empty:,})")
print(f"  - Phần tử thô: {total_raw_elem:,} | Độc bản sau Dedup: {total_dedup_elem:,}")
print(f"  - Tỷ lệ giảm trùng lặp (Dedup): -{dedup_ratio}% (tiết kiệm {dedup_saved:,} phần tử)")
print(f"  - Tổng số trích dẫn BibTeX: {total_bib:,} mục")
print(f"  - Tổng dung lượng lưu trữ: {total_storage:.1f} MB")
print('='*55)

## Bước 5: Vẽ Biểu Đồ Thống Kê Toàn Nhóm

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

os.makedirs('figures', exist_ok=True)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# --- Biểu đồ 1: Phân bố 11 loại phần tử toàn nhóm ---
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)
sorted_kinds = sorted(combined_kind_counts.items(), key=lambda x: x[1])
kinds = [k[0] for k in sorted_kinds]
counts = [k[1] for k in sorted_kinds]

bars = ax.barh(kinds, counts, color='#2563eb', edgecolor='#1e40af', alpha=0.85)
for bar in bars:
    w = bar.get_width()
    pct = w / total_dedup_elem * 100 if total_dedup_elem > 0 else 0
    ax.text(w + (max(counts)*0.015), bar.get_y() + bar.get_height()/2, f'{w:,.0f} ({pct:.1f}%)',
            va='center', ha='left', fontsize=8.5, color='#1f2937')

ax.set_title(f'Phân bố 11 Loại Phần tử Cây Phân cấp Toàn Nhóm ({total_pubs:,} bài báo)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Số lượng Phần tử Độc bản (elements)', fontsize=10)
ax.set_xlim(0, max(counts) * 1.25)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
elem_chart_path = 'figures/m2-hoang-elements.png'
plt.savefig(elem_chart_path)
plt.show()
print(f'✅ Đã lưu biểu đồ phần tử: {elem_chart_path}')

# --- Biểu đồ 2: So sánh hiệu quả Khử trùng lặp (Dedup) theo từng Owner ---
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=300)
df_ob = pd.DataFrame(owner_breakdown)
x = np.arange(len(df_ob))
width = 0.35

rects1 = ax.bar(x - width/2, df_ob['raw_elements'] / 1e6, width, label='Trước khử trùng (Thô)', color='#ef4444', alpha=0.85)
rects2 = ax.bar(x + width/2, df_ob['dedup_elements'] / 1e6, width, label='Sau khử trùng (Độc bản)', color='#10b981', alpha=0.85)

for i, row in df_ob.iterrows():
    ax.text(i, max(row['raw_elements'], row['dedup_elements']) / 1e6 + 0.05,
            f"-{row['dedup_ratio_pct']}%", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#047857')

ax.set_title('Hiệu quả Khử trùng lặp Phần tử Nội dung giữa các Phiên bản theo từng Owner', fontsize=11, fontweight='bold', pad=12)
ax.set_ylabel('Số lượng Phần tử (Triệu phần tử)', fontsize=9.5)
ax.set_xticks(x)
ax.set_xticklabels([o.upper() for o in df_ob['owner']], fontsize=9.5)
ax.legend(frameon=True, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
ver_chart_path = 'figures/m2-hoang-versions.png'
plt.savefig(ver_chart_path)
plt.show()
print(f'✅ Đã lưu biểu đồ so sánh: {ver_chart_path}')


## Bước 6: Đóng Gói và Tự Động Tải về Gói Dữ Liệu Nhẹ (~1–2 MB)

In [ ]:
import os
import json
import shutil
import zipfile
from google.colab import files

# Tạo cấu trúc thư mục xuất
os.makedirs('export/data/hoang', exist_ok=True)
os.makedirs('export/book/figures', exist_ok=True)

# 1. Ghi JSON tóm tắt toàn nhóm
json_out = 'export/data/hoang/m2_hierarchy_summary.json'
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(group_summary, f, ensure_ascii=False, indent=2)

# 2. Ghi CSV toàn nhóm
csv_out = 'export/data/hoang/m2_hierarchy_publications.csv'
df_all_pubs.to_csv(csv_out, index=False, encoding='utf-8')

# 3. Sao chép biểu đồ
shutil.copy(elem_chart_path, 'export/book/figures/m2-hoang-elements.png')
shutil.copy(ver_chart_path, 'export/book/figures/m2-hoang-versions.png')

# 4. Đóng gói zip
zip_path = 'm2_hoang_group_summary.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, filenames in os.walk('export'):
        for filename in filenames:
            fpath = os.path.join(root, filename)
            arcname = os.path.relpath(fpath, 'export')
            z.write(fpath, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'\n🎉 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {zip_path} ({zip_size_mb:.2f} MB)')
print('Đang tự động tải về máy bạn...')
files.download(zip_path)
